In [4]:
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import time
import json
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import gensim.downloader as api

In [27]:
%load_ext autoreload
%autoreload 2

from test import *
from Data.Tool import *
from Model.Tool import *
from Model.model import *

VECTOR_SIZE  = 300     # dimension des embeddings FastText
HIDDEN_DIM   = 128
BIDIRECTIONAL= True
DROPOUT      = 0.3
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 1e-3
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED         = 42
MAX_LEN      = 26

torch.manual_seed(SEED)
np.random.seed(SEED)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
df = load_and_prepare(
        corpus_path = "Data/train/corpus.tache1.learn.utf8",
        pkl_path    = "Data/train/sequences_fasttext_fr.pkl",
        vector_size = 300,
    )
print(f"Dataset chargé : {len(df)} lignes | Labels : {df['Label'].value_counts().to_dict()}")

train_df, val_df = train_test_split( df, test_size=0.2, random_state=SEED, stratify=df["Label"])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(val_df)}")

📂 Chargement du corpus...
   → 57413 lignes chargées
🧹 Nettoyage du texte...
📦 Chargement des séquences depuis 'Data/train/sequences_fasttext_fr.pkl'...
   → 57413 séquences fusionnées
🗑️  Suppression des lignes vides...
   → 35 lignes supprimées | 57378 lignes restantes
🔍 Vérification cohérence tokens ↔ vecteurs...


Vérification: 100%|██████████| 57378/57378 [00:01<00:00, 33034.79it/s]



✅ Lignes finales       : 57378
✅ Balance C/M         : {'C': 49855, 'M': 7523}
✅ Erreurs restantes  : 0
🎉 Pipeline prêt pour l'entraînement !
Dataset chargé : 57378 lignes | Labels : {'C': 49855, 'M': 7523}
Train : 45902 | Val : 11476 | Test : 11476


In [9]:
class RNN(nn.Module):
    def __init__(self, vector_size, hidden_dim, bi=False, dropout=0.2):
        super().__init__()
        self.rnn = nn.GRU(
            input_size=vector_size,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bi,
            num_layers=2,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * (1 + bi), 1)

    def forward(self, x):
        rnn_out, _ = self.rnn(x)
        pooled = torch.max(rnn_out, dim=1)[0]
        return self.fc(self.dropout(pooled)).squeeze(-1)


In [15]:
# ── Hyperparamètres ──────────────────────────────────────────
VECTOR_SIZE   = 300
HIDDEN_DIM    = 128
BIDIRECTIONAL = True
DROPOUT       = 0.3
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 3e-4
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED          = 42
torch.manual_seed(SEED)

# ── Splits ───────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["Label"])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df["Label"])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

# ── Dataloaders ──────────────────────────────────────────────
train_loader = DataLoader(SentenceDatasetContext(train_df, max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentenceDatasetContext(val_df,   max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SentenceDatasetContext(test_df,  max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

# ── Modèle ───────────────────────────────────────────────────
model     = RNN(VECTOR_SIZE, HIDDEN_DIM, bi=BIDIRECTIONAL, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

n_C = (train_df["Label"] == "C").sum()
n_M = (train_df["Label"] == "M").sum()
pos_weight = torch.tensor([n_M / n_C], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Boucle d'entraînement ────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    # -- Train
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss    += loss.item() * len(y)
        tr_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
        tr_total   += len(y)

    # -- Val
    model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y   = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            vl_loss    += loss.item() * len(y)
            vl_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
            vl_total   += len(y)

    tr_loss /= tr_total
    vl_loss /= vl_total

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss={tr_loss:.4f} Acc={tr_correct/tr_total:.4f} | Val Loss={vl_loss:.4f} Acc={vl_correct/vl_total:.4f}")


# ── Evaluation finale sur test ───────────────────────────────
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X      = X.to(DEVICE)
        logits = model(X)
        preds  = (torch.sigmoid(logits) > 0.5).float().cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

label_map  = {1.0: "C", 0.0: "M"}
pred_names = [label_map[p] for p in all_preds]
true_names = [label_map[l] for l in all_labels]

print(classification_report(true_names, pred_names, target_names=["M", "C"]))

Train : 43893 | Val : 4878 | Test : 8607
Epoch 01/20 | Train Loss=0.1594 Acc=0.6432 | Val Loss=0.1635 Acc=0.8346
Epoch 02/20 | Train Loss=0.1337 Acc=0.7813 | Val Loss=0.1539 Acc=0.8446
Epoch 03/20 | Train Loss=0.1275 Acc=0.7881 | Val Loss=0.1419 Acc=0.6704
Epoch 04/20 | Train Loss=0.1241 Acc=0.7975 | Val Loss=0.1394 Acc=0.8014
Epoch 05/20 | Train Loss=0.1209 Acc=0.8040 | Val Loss=0.1454 Acc=0.8571
Epoch 06/20 | Train Loss=0.1175 Acc=0.8108 | Val Loss=0.1432 Acc=0.8448
Epoch 07/20 | Train Loss=0.1144 Acc=0.8173 | Val Loss=0.1581 Acc=0.8815
Epoch 08/20 | Train Loss=0.1117 Acc=0.8203 | Val Loss=0.1309 Acc=0.7507
Epoch 09/20 | Train Loss=0.1092 Acc=0.8243 | Val Loss=0.1385 Acc=0.8516
Epoch 10/20 | Train Loss=0.1057 Acc=0.8350 | Val Loss=0.1479 Acc=0.8672
Epoch 11/20 | Train Loss=0.1040 Acc=0.8379 | Val Loss=0.1519 Acc=0.8657
Epoch 12/20 | Train Loss=0.1023 Acc=0.8427 | Val Loss=0.1387 Acc=0.8520
Epoch 13/20 | Train Loss=0.0996 Acc=0.8447 | Val Loss=0.1338 Acc=0.8319
Epoch 14/20 | Train Los

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNNContext(nn.Module):
    def __init__(self, vector_size, max_len, n_classes=1, dropout=0.5):
        super(TextCNNContext, self).__init__()
        
        # Le contexte fait 3 * max_len en hauteur
        self.input_height = 3 * max_len 
        
        # Canaux de sortie pour les filtres (kernels)
        out_channels = 100 
        
        # Nous utilisons des tailles de filtres variées pour capturer des n-grammes différents
        # (3, 4, 5) sont des standards pour le texte
        self.conv1 = nn.Conv2d(1, out_channels, (3, vector_size))
        self.conv2 = nn.Conv2d(1, out_channels, (4, vector_size))
        self.conv3 = nn.Conv2d(1, out_channels, (5, vector_size))
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(3 * out_channels, n_classes)

    def forward(self, x):
        # x shape: [batch_size, 3 * max_len, vector_size]
        # On ajoute une dimension pour le canal de "couleur" (comme une image NB)
        x = x.unsqueeze(1) # [batch_size, 1, 3*max_len, vector_size]

        # Convolution + Activation + MaxPool global sur la hauteur
        x1 = F.relu(self.conv1(x)).squeeze(3) # [batch, out_channels, height']
        x1 = F.max_pool1d(x1, x1.size(2)).squeeze(2) # [batch, out_channels]

        x2 = F.relu(self.conv2(x)).squeeze(3)
        x2 = F.max_pool1d(x2, x2.size(2)).squeeze(2)

        x3 = F.relu(self.conv3(x)).squeeze(3)
        x3 = F.max_pool1d(x3, x3.size(2)).squeeze(2)

        # Concaténation des différentes échelles de filtres
        combined = torch.cat((x1, x2, x3), dim=1) # [batch, 3 * out_channels]
        
        logits = self.fc(self.dropout(combined))
        return logits.view(-1) # On aplatit pour correspondre aux labels (BCEWithLogitsLoss)

In [20]:
# ── Hyperparamètres ──────────────────────────────────────────
VECTOR_SIZE   = 300
HIDDEN_DIM    = 128
BIDIRECTIONAL = True
DROPOUT       = 0.3
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 3e-4
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED          = 42
torch.manual_seed(SEED)

# ── Splits ───────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["Label"])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df["Label"])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

# ── Dataloaders ──────────────────────────────────────────────
train_loader = DataLoader(SentenceDatasetContext(train_df, max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentenceDatasetContext(val_df,   max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SentenceDatasetContext(test_df,  max_len=MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

# ── Modèle ───────────────────────────────────────────────────
model     = TextCNNContext(VECTOR_SIZE,MAX_LEN)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

n_C = (train_df["Label"] == "C").sum()
n_M = (train_df["Label"] == "M").sum()
pos_weight = torch.tensor([n_M / n_C], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Boucle d'entraînement ────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    # -- Train
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss    += loss.item() * len(y)
        tr_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
        tr_total   += len(y)

    # -- Val
    model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y   = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            vl_loss    += loss.item() * len(y)
            vl_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
            vl_total   += len(y)

    tr_loss /= tr_total
    vl_loss /= vl_total

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss={tr_loss:.4f} Acc={tr_correct/tr_total:.4f} | Val Loss={vl_loss:.4f} Acc={vl_correct/vl_total:.4f}")


# ── Evaluation finale sur test ───────────────────────────────
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X      = X.to(DEVICE)
        logits = model(X)
        preds  = (torch.sigmoid(logits) > 0.5).float().cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

label_map  = {1.0: "C", 0.0: "M"}
pred_names = [label_map[p] for p in all_preds]
true_names = [label_map[l] for l in all_labels]

print(classification_report(true_names, pred_names, target_names=["M", "C"]))

Train : 43893 | Val : 4878 | Test : 8607
Epoch 01/20 | Train Loss=0.1645 Acc=0.6266 | Val Loss=0.1600 Acc=0.7140
Epoch 02/20 | Train Loss=0.1464 Acc=0.7347 | Val Loss=0.1550 Acc=0.7825
Epoch 03/20 | Train Loss=0.1350 Acc=0.7656 | Val Loss=0.1508 Acc=0.7956
Epoch 04/20 | Train Loss=0.1270 Acc=0.7818 | Val Loss=0.1443 Acc=0.8079
Epoch 05/20 | Train Loss=0.1200 Acc=0.7982 | Val Loss=0.1385 Acc=0.7259
Epoch 06/20 | Train Loss=0.1140 Acc=0.8079 | Val Loss=0.1378 Acc=0.7347
Epoch 07/20 | Train Loss=0.1068 Acc=0.8208 | Val Loss=0.1441 Acc=0.8444
Epoch 08/20 | Train Loss=0.1019 Acc=0.8300 | Val Loss=0.1356 Acc=0.8219
Epoch 09/20 | Train Loss=0.0966 Acc=0.8395 | Val Loss=0.1394 Acc=0.8282
Epoch 10/20 | Train Loss=0.0904 Acc=0.8521 | Val Loss=0.1311 Acc=0.8003
Epoch 11/20 | Train Loss=0.0860 Acc=0.8608 | Val Loss=0.1322 Acc=0.8007
Epoch 12/20 | Train Loss=0.0799 Acc=0.8720 | Val Loss=0.1294 Acc=0.7792
Epoch 13/20 | Train Loss=0.0748 Acc=0.8818 | Val Loss=0.1288 Acc=0.8175
Epoch 14/20 | Train Los

In [22]:
import json
import torch
from sklearn.metrics import classification_report, precision_recall_fscore_support

# --- Calcul des métriques finales pour le JSON ---
# On récupère precision, recall, f1 pour chaque classe et la moyenne
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
report_dict = classification_report(all_labels, all_preds, output_dict=True)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- Construction du dictionnaire JSON ---
resultats_complets = {
    "config": {
        "nom_modele": "RNN_Bidirectional" if BIDIRECTIONAL else "RNN_Simple",
        "representation": "Word2Vec_or_Custom", # À adapter selon tes embeddings
        "nb_parametres_totaux": int(count_parameters(model)),
        "taille_vocabulaire_entree": VECTOR_SIZE, 
        "nb_folds": 1,  # Ici c'est un simple split Train/Val/Test
        "epochs_par_fold": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "device_utilise": str(DEVICE),
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT
    },
    "temps_execution": {
        "total_sec": "N/A", # Tu peux ajouter un timer si besoin
        "moyen_par_fold_sec": "N/A"
    },
    "performances_moyennes": {
        "accuracy": {
            "mean": round(report_dict['accuracy'], 4),
            "std": 0.0
        },
        "f1-score": {
            "mean": round(f1, 4),
            "std": 0.0
        },
        "precision": {
            "mean": round(precision, 4),
            "std": 0.0
        },
        "recall": {
            "mean": round(recall, 4),
            "std": 0.0
        }
    },
    "details_par_split": {
        "test_report": report_dict
    }
}

# Dans ton bloc de sauvegarde JSON :
resultats_complets["config"]["nom_modele"] = "TextCNN_Context"
resultats_complets["config"]["nb_filtres"] = 300 # 100 * 3 tailles
resultats_complets["config"]["filter_sizes"] = [3, 4, 5]

# --- Sauvegarde ---
filename = "results_cnn_context3.json"
with open(filename, 'w') as f:
    json.dump(resultats_complets, f, indent=4)

print(f"✅ Rapport RNN sauvegardé : {filename}")

✅ Rapport RNN sauvegardé : results_cnn_context3.json


In [ ]:
# ── Hyperparamètres ──────────────────────────────────────────
VECTOR_SIZE   = 300
HIDDEN_DIM    = 128
BIDIRECTIONAL = True
DROPOUT       = 0.3
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 3e-4
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED          = 42
torch.manual_seed(SEED)

# ── Splits ───────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["Label"])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df["Label"])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

# ── Dataloaders ──────────────────────────────────────────────
train_loader = DataLoader(SentenceDatasetMobilContext(train_df, max_len=MAX_LEN,window_size=3), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentenceDatasetMobilContext(val_df,   max_len=MAX_LEN,window_size=3), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SentenceDatasetMobilContext(test_df,  max_len=MAX_LEN,window_size=3), batch_size=BATCH_SIZE, shuffle=False)

# ── Modèle ───────────────────────────────────────────────────
model     = TextCNNContext(VECTOR_SIZE,MAX_LEN)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

n_C = (train_df["Label"] == "C").sum()
n_M = (train_df["Label"] == "M").sum()
pos_weight = torch.tensor([n_M / n_C], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Boucle d'entraînement ────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    # -- Train
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss    += loss.item() * len(y)
        tr_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
        tr_total   += len(y)

    # -- Val
    model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y   = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            vl_loss    += loss.item() * len(y)
            vl_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
            vl_total   += len(y)

    tr_loss /= tr_total
    vl_loss /= vl_total

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss={tr_loss:.4f} Acc={tr_correct/tr_total:.4f} | Val Loss={vl_loss:.4f} Acc={vl_correct/vl_total:.4f}")


# ── Evaluation finale sur test ───────────────────────────────
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X      = X.to(DEVICE)
        logits = model(X)
        preds  = (torch.sigmoid(logits) > 0.5).float().cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

label_map  = {1.0: "C", 0.0: "M"}
pred_names = [label_map[p] for p in all_preds]
true_names = [label_map[l] for l in all_labels]

print(classification_report(true_names, pred_names, target_names=["M", "C"]))

In [30]:
import json
import torch
from sklearn.metrics import classification_report, precision_recall_fscore_support

# --- Calcul des métriques finales pour le JSON ---
# On récupère precision, recall, f1 pour chaque classe et la moyenne
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
report_dict = classification_report(all_labels, all_preds, output_dict=True)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- Construction du dictionnaire JSON ---
resultats_complets = {
    "config": {
        "nom_modele": "RNN_Bidirectional" if BIDIRECTIONAL else "RNN_Simple",
        "representation": "Word2Vec_or_Custom", # À adapter selon tes embeddings
        "nb_parametres_totaux": int(count_parameters(model)),
        "taille_vocabulaire_entree": VECTOR_SIZE, 
        "nb_folds": 1,  # Ici c'est un simple split Train/Val/Test
        "epochs_par_fold": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "device_utilise": str(DEVICE),
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT
    },
    "temps_execution": {
        "total_sec": "N/A", # Tu peux ajouter un timer si besoin
        "moyen_par_fold_sec": "N/A"
    },
    "performances_moyennes": {
        "accuracy": {
            "mean": round(report_dict['accuracy'], 4),
            "std": 0.0
        },
        "f1-score": {
            "mean": round(f1, 4),
            "std": 0.0
        },
        "precision": {
            "mean": round(precision, 4),
            "std": 0.0
        },
        "recall": {
            "mean": round(recall, 4),
            "std": 0.0
        }
    },
    "details_par_split": {
        "test_report": report_dict
    }
}

# Dans ton bloc de sauvegarde JSON :
resultats_complets["config"]["nom_modele"] = "TextCNN_Context10"
resultats_complets["config"]["nb_filtres"] = 300 # 100 * 3 tailles
resultats_complets["config"]["filter_sizes"] = [3, 4, 5]

# --- Sauvegarde ---
filename = "results_cnn_context10.json"
with open(filename, 'w') as f:
    json.dump(resultats_complets, f, indent=4)

print(f"✅ Rapport RNN sauvegardé : {filename}")

✅ Rapport RNN sauvegardé : results_cnn_context10.json


In [ ]:
# ── Hyperparamètres ──────────────────────────────────────────
VECTOR_SIZE   = 300
HIDDEN_DIM    = 128
BIDIRECTIONAL = True
DROPOUT       = 0.3
BATCH_SIZE    = 32
EPOCHS        = 20
LR            = 3e-4
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED          = 42
torch.manual_seed(SEED)

# ── Splits ───────────────────────────────────────────────────
train_df, test_df = train_test_split(df, test_size=0.15, random_state=SEED, stratify=df["Label"])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=SEED, stratify=train_df["Label"])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

# ── Dataloaders ──────────────────────────────────────────────
train_loader = DataLoader(SentenceDatasetMobilContext(train_df, max_len=MAX_LEN,window_size=3), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SentenceDatasetMobilContext(val_df,   max_len=MAX_LEN,window_size=3), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(SentenceDatasetMobilContext(test_df,  max_len=MAX_LEN,window_size=3), batch_size=BATCH_SIZE, shuffle=False)

# ── Modèle ───────────────────────────────────────────────────
model     = TextCNNContext(VECTOR_SIZE,MAX_LEN)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

n_C = (train_df["Label"] == "C").sum()
n_M = (train_df["Label"] == "M").sum()
pos_weight = torch.tensor([n_M / n_C], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# ── Boucle d'entraînement ────────────────────────────────────
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):

    # -- Train
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        tr_loss    += loss.item() * len(y)
        tr_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
        tr_total   += len(y)

    # -- Val
    model.eval()
    vl_loss, vl_correct, vl_total = 0.0, 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y   = X.to(DEVICE), y.to(DEVICE)
            logits = model(X)
            loss   = criterion(logits, y)
            vl_loss    += loss.item() * len(y)
            vl_correct += ((torch.sigmoid(logits) > 0.5).float() == y).sum().item()
            vl_total   += len(y)

    tr_loss /= tr_total
    vl_loss /= vl_total

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss={tr_loss:.4f} Acc={tr_correct/tr_total:.4f} | Val Loss={vl_loss:.4f} Acc={vl_correct/vl_total:.4f}")


# ── Evaluation finale sur test ───────────────────────────────
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in test_loader:
        X      = X.to(DEVICE)
        logits = model(X)
        preds  = (torch.sigmoid(logits) > 0.5).float().cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y.numpy())

label_map  = {1.0: "C", 0.0: "M"}
pred_names = [label_map[p] for p in all_preds]
true_names = [label_map[l] for l in all_labels]

print(classification_report(true_names, pred_names, target_names=["M", "C"]))